# Budget and Policy

`Budget` and `Policy` control cost, latency, inference caps, and PII handling during normalization. Defaults are deterministic and safe.

- **Budget:** `max_total_cost_usd`, `max_total_latency_ms`, `max_remote_inference_calls`, `max_capability_invocations` — caps on cost, wall-clock latency, and capability invocations.
- **Policy:** `unresolved_acceptable`, `currency_policy`, `log_raw_input`, `record_inference_io`, `embed_evidence_payload`, `emit_metrics`, `allow_remote_inference`, `allow_local_inference`, `confidence_floor` — behavior and privacy controls.

In [ ]:
from decimal import Decimal

from pydantic import BaseModel

import paxman
import paxman.contract.adapters.pydantic


class Product(BaseModel):
    name: str
    price: Decimal
    sku: str


# First, show the defaults
print("Default Budget:", paxman.Budget())
print("Default Policy:", paxman.Policy())
print()

In [ ]:
# Run with a tiny latency budget that forces UNRESOLVED for some fields
sample = "Widget A, price $12.99, SKU: WDG-001"

tight = paxman.normalize(
    sample,
    Product,
    budget=paxman.Budget(max_total_latency_ms=1),  # unrealistically tight
    policy=paxman.Policy(log_raw_input=False),
)
print(f"Status: {tight.status.name}")
print(f"Unresolved fields: {tight.unresolved_fields}")
print(
    f"Diagnostics: {tight.diagnostics[:3]}..."
    if len(tight.diagnostics) > 3
    else f"Diagnostics: {tight.diagnostics}"
)
print()

# Now run with default budget
default = paxman.normalize(sample, Product, policy=paxman.Policy(log_raw_input=False))
print(f"Default status: {default.status.name}")
print(f"Default unresolved: {default.unresolved_fields}")
print(f"Default data: {default.normalized_data}")

In [ ]:
# Determinism: same contract + same input = same hash
run1 = paxman.normalize(
    "Widget A, price $12.99", Product, policy=paxman.Policy(log_raw_input=False)
)
run2 = paxman.normalize(
    "Widget A, price $12.99", Product, policy=paxman.Policy(log_raw_input=False)
)
print(f"run1 hash: {run1.replay_hash}")
print(f"run2 hash: {run2.replay_hash}")
print(f"Deterministic: {run1.replay_hash == run2.replay_hash}")

## Key settings

**Budget fields:**
- `max_total_cost_usd` — hard cap on cost in USD (Decimal, per ADR-0004 / ADR-0010); aborts when exceeded. Default: no cap.
- `max_total_latency_ms` — hard cap on wall-clock latency. Default: no cap.
- `max_remote_inference_calls` — cap on remote-inference invocations. Default: no cap.
- `max_capability_invocations` — cap on total capability invocations. Default: no cap.

**Policy fields:**
- `log_raw_input` — set to `False` (default) to prevent raw PII appearing in logs.
- `currency_policy` — how to handle cross-currency MONEY candidates (default: `STRICT_MATCH`).
- `unresolved_acceptable` — if `True`, allow `PARTIAL_SUCCESS` when required fields are unresolved. Default: `False`.
- `allow_remote_inference` / `allow_local_inference` — gate inference tiers. Default: both `True`.
- `confidence_floor` — minimum confidence to mark a field `SUCCESS`; below this is `PARTIAL_SUCCESS`. Default: `0.80`.
- `record_inference_io` / `embed_evidence_payload` / `emit_metrics` — opt-in for richer artifacts. Default: all `False`.

> **Tip:** Always keep `log_raw_input=False` in production to avoid leaking PII into structured logs.

## Try it yourself

- Compare artifact hashes across different `Budget` values — they should differ because the execution path changes.
- Set `log_raw_input=True` and inspect the logs from `make playground-logs`.
- Set `unresolved_acceptable=True` and observe the status when fields are missing.